# Stack and especially Monotonic Stack

In [2]:
from typing import List, Optional

<b> 316 Remove Duplicate Letters </b>

the result is smallest in lexicographical order, hence using monotonic stack

time complexity:
-   for loop O(N)
-   inside of loop, the checking of c in stack takes at most O(N) depending on stack size
-   the while loop at most will take O(N) depending on stack size
-   so in total O(N*N)

In [3]:
class Solution:
    def removeDuplicateLetters(self, s: str) -> str:
        ind_dict = {c: i for i, c in enumerate(s)}
        stack = [] # save char
        for i, c in enumerate(s):
            if c in stack: continue
            # if prev char > curr char and this prev char will appear 
            # later in the string, it is safe to pop it out because 
            # it will be added later.
            while stack and stack[-1] > c and ind_dict[stack[-1]]>i:
                stack.pop()
            stack.append(c)
        return "".join(stack)

<b> 739 Daily Temperature </b>

my intuition is to loop the arrary from right to left and keep track of the max temperature and its index along the way, but it is not right. I dont need to compare each pos with the max temperature, but the next larger temperature. 

so here a monotonic stack is better.

the stack stores index, with corresponding temperate in descending order. When checking an index, see if any previous visited index needs to update the res[i] value.

In [ ]:

def dailyTemperatures(T: List[int]) -> List[int]:
    n = len(T)
    stack = [] # save index
    res = [0]*n
    
    # find next larger value, get the distance
    for i, v in enumerate(T):
        # remember compare the actual value in T with V
        while stack and T[stack[-1]]<v:
            ind = stack.pop()
            res[ind] = i-ind
        stack.append(i)
    return res
    

<b> 496 next greater element I </b>

using monotonic stack to construct a dictionary with {val: next_greater_val} mapping. When looping over the nums2 array, keep track of the stack in descending order. Whenever a new item visited is greater than the value in stack, the value in stack found its next greater value.

time complexity:
-   loop takes O(N), the value in nums2 at most visited twice, one in for loop another in while loop, so the total time is still O(N)
-   hashmap query is O(1) for each item in nums1.
-   in total O(M+N) M+N are the size of the two arrays

In [4]:

def nextGreaterElement(nums1: List[int], nums2: List[int]) -> List[int]:
    """
    all number in nums1 and nums2 are valid
    can construct the res list for each val in nums2 to find the next greater value, 
    then query using the value in nums1
    """
    res = {} # value: next_greater_val
    stack = [] # monotonic stack, descending order
    for val in nums2:
        while stack and stack[-1] < val:
            item = stack.pop()
            res[item] = val
        stack.append(val)
    
    ans = [res.get(item, -1) for item in nums1]
    return ans
    

<b> 503 Next greater element II </b>

the array is in circular. unless all values are the same, there will always be a next greater value.

looping the array twice will ensure the next greater value is found.

In [5]:

def nextGreaterElements(nums: List[int]) -> List[int]:
    """
    time complexity: main for loop is O(2N) = O(N), the prev_ind_stack only has max len
    N, since each element can only be pushed and popped once. so this while loop is O(N) 
    in total across all for loop iteration. 
    so total time complexity is O(N) + O(N) = O(N)
    """
    n = len(nums)
    res = [-1]*n
    prev_ind_stack = []

    # loop over the nums list twice is enough to generate the results
    for i in range(2*n):
        val = nums[i%n]

        while prev_ind_stack and nums[prev_ind_stack[-1]] < val:
            # current value is greater than the previous visited val at index stored in the stack
            # that means the previous index find its next higher value
            # keep popping the ind from the stack, because current val can be the next higher value for multiple previous index
            res[prev_ind_stack.pop()] = val

        # a stack to keep previously visited index
        if i < n:
            prev_ind_stack.append(i)
    return res
        

<b> 556 Next greater element III </b>

looks like finding the next value in lexicagraphic order.

step 1: from right to left, find the first position where nums[i-1] < nums[i], i-1 is the position we should switch with the value found in step 2:  

step 2: from right to left, find the first value greater than nums[i-1] (it is actually the smallest greater element from the right because the right section is in ascending order from right to left, so the first one found larger than nums[i-1] is the smallest greater element found), that is the value to switch with nums[i-1] to make the next larger value in lexicographic order.

step 3: need to ensure the value from nums[i:] is in ascending order, which helps to make the value the next larger value in lexicographic order.

given 32 bit integer limit, how to know what is the largest value:
- 32 bit, 1 bit is used to track the sign: genative or positive
- 31 bit is used to track the value, largest value is when all the bit is 1, so 2^31 -1, -1 because need to save one space for value 0
    -   With 31 bits, the largest possible number that can be represented is when all the bits are set to 1
    -   2 ** 30 + 2 ** 29 + ... + 2 ** 0 = 2 ** 31 - 1 -> formula


In [ ]:
class Solution:
    def nextGreaterElement(self, n: int) -> int:
        n_array = list(map(int, str(n))) # into a list of int
        i = len(n_array)-1
        # find first number from right to left breaking the ascending order
        while i-1>=0 and n_array[i-1]>=n_array[i]:
            i-=1
        # case 0: all digits in descending order from left to right, no result
        if i==0:
            return -1
        
        # find the first digit from right to left that is greater than the digit in position i-1
        j = len(n_array)-1
        while j>i-1 and n_array[j]<=n_array[i-1]:
            j-=1
        # now j+1 points to the first digit that will be <= n_array[i-1], so n_array[j] is the smallest larger digit for n_array[i-1], swap the two
        n_array[i-1], n_array[j] = n_array[j], n_array[i-1]
        # then reorder digits from index [i, n-1]. it takes care of case 2: all numers are ascending, swap the last two digits and case 3: other cases
        n_array[i:] = sorted(n_array[i:])
        res = int(''.join(map(str, n_array)))
        return res if res <= (2**31-1) else -1
        

        

<b> 2454 Next Greater element IV </b>

<b> 901 online stock span </b>

monotonic stack too, but the stack also saves the span for the current stock price

In [6]:
class StockSpanner:

    def __init__(self):
        # self.stock_span = {} # day: span
        self.stack=[] # (price, span), save stock price in decending order

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            _, prev_span = self.stack.pop()
            span += prev_span
        self.stack.append((price, span))
        return span
        


# Your StockSpanner object will be instantiated and called as such:
# obj = StockSpanner()
# param_1 = obj.next(price)

# Normal Stack problem

<b> 636 Exclusive time of functions </b>

using a stack to save "start" function id, using a var end_time to keep track of the most recent end_time which is the start time of next calcuation.

[start, start, end, start, end, end] -> for third start, the time between n[2] and n[3] should be added to n[0], the start that hasnt closed yet. 

In [7]:

def exclusiveTime(n: int, logs: List[str]) -> List[int]:
    res = [0]*n
    start_stack = [] # f_id, non-paired start id
    end_time = 0 # the start time of next calcuation.
    for l in logs:
        f_id, sign,t = l.split(":")
        f_id = int(f_id)
        t = int(t)
        # assuming CPU work correctly, end is followed by start for same id, wont have case like 0:start, 1:start, 0:end, 1:end
        if sign == "start":
            if start_stack:
                prev_f_id = start_stack[-1]
                # start - start: 
                res[prev_f_id] += t - end_time
            start_stack.append(f_id)
            end_time = t

        elif sign == "end":
            # assuming there must be a start previously
            start_stack.pop()
            # end - start, end should be included, hence + 1
            res[f_id] += t-end_time+1
            # end+1 becomes the start of next interval
            end_time = t+1

    return res        

<b> 84 largest rectangle in histogram </b>


smart to transform into a monotone stack problem, plus the DP way to quickly find the answer

In [8]:

def largestRectangleArea(heights: List[int]) -> int:
    """
    for ith position, using the heights[i] to calculate the size
    the max rectangle size is determined by the width it can get: r - l -1
    where r represents the rightest location with h[r] >= h[i]
    because if h[r] < h[i], position r should NOT be considered as part of the rectangle
    same, l represents the leftest location with h[l] >= h[i]
    if h[l]<h[i], position l should NOT be considered as well

    so each i, we should know its right most and left most boundary that can use
    h[i] as the height to calculate the rectangle size
    """
    if not heights:
        return 0
    n = len(heights)
    max_area = 0
    # left_most[i] mean for ith position, the left most index position can reach
    left_most = [0] * n # initial position for left
    right_most = [0] * n # initial position for right
    left_most[0] = -1
    right_most[-1] = n
    for i in range(1, n):
        p = i-1
        while p >=0 and heights[p] >=heights[i]:
            # for p position, left_most[p] is the first left border where heights[left_most[p]] < heights[p]
            # heights[left_most[p]+1] >= heights[p]
            p = left_most[p]
        left_most[i] = p
    for i in range(n-2, -1, -1):
        p = i+1 # start from n-1
        while p < n and heights[p] >= heights[i]:
            p = right_most[p]
        right_most[i] = p
    
    for i in range(n):
        max_area = max(max_area, heights[i] * (right_most[i]-left_most[i]-1))
    return max_area

<b> 394 Decode String </b>

using num stack and a string stack. different case: str, num, [, ], respond differently.

In [13]:

def decodeString(s: str) -> str:
    # save num
    num_stack = []
    char_stack = []
    # save char
    num = 0
    tmp = ""
    for i in s:
        if i.isnumeric():
            num = num*10 + int(i)
        elif i == "[":
            # num should be reset for next num in the list
            num_stack.append(num)
            char_stack.append(tmp)
            num = 0
            tmp = ""
        elif i.isalpha():
            tmp += i
        elif i == "]":
            # pre_str always exist because stack will append an empty string anyway, to avoid pop from empty stack
            # example 3[a] when saw [, the char_stack will append "" empty string
            # pop out one item from num_stack and one item_from char_stack for decoding
            n = num_stack.pop() # dont use variable num
            prev_str = char_stack.pop()
            tmp = prev_str + n * tmp
    return tmp

In [12]:
decodeString("3[a]")

0 a


'aaa'